In [ ]:
import pandas as pd
import numpy as np
import os
import sys

from model_metrics import summarize_model_performance
from model_tuner import loadObjects

sys.path.append("../")

from core.functions import (
    normalize_actor,
    build_actor_interaction_graph,
    add_pairwise_embedding_features,
)

In [ ]:
model_xgb = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/453650780489761177/f14eeb2cdce94d128c5525f7dfabc096/artifacts/xgb_log_fatalities/model.pkl"
)

In [ ]:
model_xgb.estimator

In [ ]:
len(model_xgb.get_feature_names())

In [ ]:
X_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

In [ ]:
y_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_test_log_fatalities.parquet"
)

In [ ]:
y_test.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
y_test

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")

In [ ]:
log_preds["pred_fatalities"] = round(
    np.expm1(log_preds["log_pred_fatalities"]), 0
).astype(int)

In [ ]:
log_preds_merge = log_preds.join(X_test, how="inner", on="event_id_cnty")

In [ ]:
y_test_actual = pd.DataFrame(
    np.expm1(y_test["actual_log_fatalities"]).round(0).astype(int),
    columns=["actual_log_fatalities"],
).rename(columns={"actual_log_fatalities": "actual_fatalities"})

In [ ]:
y_test_actual

In [ ]:
log_preds_merge = y_test_actual.merge(log_preds_merge, how="inner", on="event_id_cnty")

In [ ]:
log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

In [ ]:
model_xgb.get_feature_names()

In [ ]:
dir(model_xgb)